In [ ]:
import datetime

#
# DATE = datetime.datetime.now()

# comment out line below and use line <bove if you want to fetch the last data

# Input data in the <YYYY-MM-DD> format in the string
DATE = datetime.datetime.fromisoformat("2026-MM-DD")

### Load api key from .env file into os.environ, then read env variable from there

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads variables from .env into os.environ

API_KEY = os.getenv("FIRM_API_KEY")
if not API_KEY:
    raise Exception("Missing FIRM_API_KEY. Set it in your environment or .env file.")

### Test api key

In [3]:
import requests

url = "https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY="
parameters = {"MAP_KEY": API_KEY}
response = requests.get(url, params=parameters)
if response.status_code == 200:
    print(f"Api request successful\n{'-' * 22}")

    # Convert the JSON response into a Python dictionary
    data = response.json()
    print(f"Current transactions: {data['current_transactions']}/5000")
else:
    raise Exception(
        f"An error occured with status code {response.status_code}: {response.reason}"
    )

Api request successful
----------------------
Current transactions: 0/5000


### Fetch data from the last 14 days

In [4]:
import time
import requests
from pathlib import Path
from tqdm import tqdm

date = DATE + datetime.timedelta(days=-1)
date_str = date.strftime("%Y-%m-%d")
data = []

for _i in tqdm(range(0, 7)):
    url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        + API_KEY
        + "/VIIRS_SNPP_NRT/world/2/"
        + date_str
    )
    date = date + datetime.timedelta(days=-2)
    date_str = date.strftime("%Y-%m-%d")
    response = requests.get(url)
    if response.status_code == 200:
        parsed_response = response.content.decode("utf-8")
        # if data is still empty, also append the column
        if not data:
            data.extend(parsed_response.split("\n"))
        else:
            data.extend(parsed_response.split("\n")[1:])
    else:
        raise Exception(
            f"An error occured with status code {response.status_code}: {response.reason}"
        )
    # api seems to behave well, but we sleep half second nonetheless
    time.sleep(0.5)

dir_path = Path(Path.cwd().parent.joinpath("data"))
if not dir_path.exists():
    Path.mkdir(dir_path)
    Path.mkdir(Path(dir_path.joinpath("raw")))
    Path.mkdir(Path(dir_path.joinpath("processed")))

# use Path object so we don't have to worry about path separator differences
file_path = Path.cwd().parent.joinpath(
    "data/raw", f"VIIRS_SNNP_NRT_world_14days_{DATE.strftime('%Y-%m-%d')}.csv"
)

with open(file_path, "w") as file:
    file.write("\n".join(data))


100%|██████████| 7/7 [00:22<00:00,  3.24s/it]


### Download world national boundaries

In [4]:
import requests
from pathlib import Path

url = "https://github.com/wmgeolab/geoBoundaries/raw/refs/heads/main/releaseData/CGAZ/geoBoundariesCGAZ_ADM0.geojson"
response = requests.get(url)
if response.status_code == 200:
    file_path = Path.cwd().parent.joinpath("data/raw", "geoboundaries_world.geojson")
    with open(file_path, "wb") as f:
        f.write(response.content)
else:
    raise Exception(
        f"An error occured with status code {response.status_code}: {response.reason}"
    )

In [2]:
import requests
from pathlib import Path

url = "https://data360files.worldbank.org/data360-data/data/WB_WDI/WB_WDI_AG_SRF_TOTL_K2.csv"
response = requests.get(url)
if response.status_code == 200:
    file_path = Path.cwd().parent.joinpath("data/raw", "surface_area.csv")
    with open(file_path, "wb") as f:
        f.write(response.content)
else:
    raise Exception(
        f"An error occured with status code {response.status_code}: {response.reason}"
    )